# Spark Advanced Features – Complex Transformations & Window Functions

- Complex transformations
- Window functions


In [2]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder     .appName("Spark Advanced Features")     .getOrCreate()

spark


## Sample Employee Dataset

In [3]:

data = [
    (1, "Amit", "IT", 60000, "2023-01-10"),
    (2, "Neha", "HR", 50000, "2023-01-15"),
    (3, "Raj", "IT", 70000, "2023-02-01"),
    (4, "Priya", "Finance", 65000, "2023-02-10"),
    (5, "Suresh", "IT", 80000, "2023-03-05"),
    (6, "Kiran", "HR", 52000, "2023-03-18")
]

columns = ["id", "name", "department", "salary", "join_date"]

df = spark.createDataFrame(data, columns).withColumn("join_date", to_date("join_date"))

df.show()


+---+------+----------+------+----------+
| id|  name|department|salary| join_date|
+---+------+----------+------+----------+
|  1|  Amit|        IT| 60000|2023-01-10|
|  2|  Neha|        HR| 50000|2023-01-15|
|  3|   Raj|        IT| 70000|2023-02-01|
|  4| Priya|   Finance| 65000|2023-02-10|
|  5|Suresh|        IT| 80000|2023-03-05|
|  6| Kiran|        HR| 52000|2023-03-18|
+---+------+----------+------+----------+



# 1️⃣ Complex Transformations

### withColumn, when, otherwise

In [ ]:

df_salary_band = df.withColumn(
    "salary_band",
    when(col("salary") >= 75000, "HIGH")
    .when(col("salary") >= 60000, "MEDIUM")
    .otherwise("LOW")
)

df_salary_band.show()


+---+------+----------+------+----------+-----------+
| id|  name|department|salary| join_date|salary_band|
+---+------+----------+------+----------+-----------+
|  1|  Amit|        IT| 60000|2023-01-10|     MEDIUM|
|  2|  Neha|        HR| 50000|2023-01-15|        LOW|
|  3|   Raj|        IT| 70000|2023-02-01|     MEDIUM|
|  4| Priya|   Finance| 65000|2023-02-10|     MEDIUM|
|  5|Suresh|        IT| 80000|2023-03-05|       HIGH|
|  6| Kiran|        HR| 52000|2023-03-18|        LOW|
+---+------+----------+------+----------+-----------+



### explode & split

split() converts a string column into an array column based on a delimiter.

explode() converts an array column into multiple rows.

In [ ]:

skills_data = [
    (1, "Python,Spark,SQL"),
    (2, "Excel,HRMS"),
    (3, "Python,Spark"),
]

skills_df = spark.createDataFrame(skills_data, ["emp_id", "skills"])

skills_df.withColumn("skill", explode(split("skills", ","))).show()


+------+----------------+------+
|emp_id|          skills| skill|
+------+----------------+------+
|     1|Python,Spark,SQL|Python|
|     1|Python,Spark,SQL| Spark|
|     1|Python,Spark,SQL|   SQL|
|     2|      Excel,HRMS| Excel|
|     2|      Excel,HRMS|  HRMS|
|     3|    Python,Spark|Python|
|     3|    Python,Spark| Spark|
+------+----------------+------+



### struct & nested columns

In [5]:

df_struct = df.withColumn("employee_info",struct("name", "department", "salary"))

df_struct.select("id", "employee_info").show(truncate=False)


+---+-----------------------+
|id |employee_info          |
+---+-----------------------+
|1  |{Amit, IT, 60000}      |
|2  |{Neha, HR, 50000}      |
|3  |{Raj, IT, 70000}       |
|4  |{Priya, Finance, 65000}|
|5  |{Suresh, IT, 80000}    |
|6  |{Kiran, HR, 52000}     |
+---+-----------------------+



### array & array functions

In [ ]:

df_array = df.withColumn("bonus_array", array(col("salary") * 0.05, col("salary") * 0.10))

df_array.show()


+---+------+----------+------+----------+----------------+
| id|  name|department|salary| join_date|     bonus_array|
+---+------+----------+------+----------+----------------+
|  1|  Amit|        IT| 60000|2023-01-10|[3000.0, 6000.0]|
|  2|  Neha|        HR| 50000|2023-01-15|[2500.0, 5000.0]|
|  3|   Raj|        IT| 70000|2023-02-01|[3500.0, 7000.0]|
|  4| Priya|   Finance| 65000|2023-02-10|[3250.0, 6500.0]|
|  5|Suresh|        IT| 80000|2023-03-05|[4000.0, 8000.0]|
|  6| Kiran|        HR| 52000|2023-03-18|[2600.0, 5200.0]|
+---+------+----------+------+----------+----------------+



# 2️⃣ Window Functions

### Window specification

In [ ]:

dept_window = Window.partitionBy("department").orderBy(col("salary").desc())


### row_number

In [ ]:

df.withColumn(
    "row_number",
    row_number().over(dept_window)
).show()


+---+------+----------+------+----------+----------+
| id|  name|department|salary| join_date|row_number|
+---+------+----------+------+----------+----------+
|  4| Priya|   Finance| 65000|2023-02-10|         1|
|  6| Kiran|        HR| 52000|2023-03-18|         1|
|  2|  Neha|        HR| 50000|2023-01-15|         2|
|  5|Suresh|        IT| 80000|2023-03-05|         1|
|  3|   Raj|        IT| 70000|2023-02-01|         2|
|  1|  Amit|        IT| 60000|2023-01-10|         3|
+---+------+----------+------+----------+----------+



### rank & dense_rank

In [ ]:

df.withColumn(
    "rank",
    rank().over(dept_window)
).withColumn(
    "dense_rank",
    dense_rank().over(dept_window)
).show()


+---+------+----------+------+----------+----+----------+
| id|  name|department|salary| join_date|rank|dense_rank|
+---+------+----------+------+----------+----+----------+
|  4| Priya|   Finance| 65000|2023-02-10|   1|         1|
|  6| Kiran|        HR| 52000|2023-03-18|   1|         1|
|  2|  Neha|        HR| 50000|2023-01-15|   2|         2|
|  5|Suresh|        IT| 80000|2023-03-05|   1|         1|
|  3|   Raj|        IT| 70000|2023-02-01|   2|         2|
|  1|  Amit|        IT| 60000|2023-01-10|   3|         3|
+---+------+----------+------+----------+----+----------+



### lead & lag

In [ ]:

salary_window = Window.partitionBy("department").orderBy("join_date")

df.withColumn(
    "prev_salary",
    lag("salary").over(salary_window)
).withColumn(
    "next_salary",
    lead("salary").over(salary_window)
).show()


+---+------+----------+------+----------+-----------+-----------+
| id|  name|department|salary| join_date|prev_salary|next_salary|
+---+------+----------+------+----------+-----------+-----------+
|  4| Priya|   Finance| 65000|2023-02-10|       NULL|       NULL|
|  2|  Neha|        HR| 50000|2023-01-15|       NULL|      52000|
|  6| Kiran|        HR| 52000|2023-03-18|      50000|       NULL|
|  1|  Amit|        IT| 60000|2023-01-10|       NULL|      70000|
|  3|   Raj|        IT| 70000|2023-02-01|      60000|      80000|
|  5|Suresh|        IT| 80000|2023-03-05|      70000|       NULL|
+---+------+----------+------+----------+-----------+-----------+



### running total (cumulative sum)

In [ ]:

running_window = Window.partitionBy("department").orderBy("join_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df.withColumn(
    "running_salary_total",
    sum("salary").over(running_window)
).show()


+---+------+----------+------+----------+--------------------+
| id|  name|department|salary| join_date|running_salary_total|
+---+------+----------+------+----------+--------------------+
|  4| Priya|   Finance| 65000|2023-02-10|               65000|
|  2|  Neha|        HR| 50000|2023-01-15|               50000|
|  6| Kiran|        HR| 52000|2023-03-18|              102000|
|  1|  Amit|        IT| 60000|2023-01-10|               60000|
|  3|   Raj|        IT| 70000|2023-02-01|              130000|
|  5|Suresh|        IT| 80000|2023-03-05|              210000|
+---+------+----------+------+----------+--------------------+



### Window aggregation without collapsing rows

In [ ]:

avg_window = Window.partitionBy("department")

df.withColumn(
    "dept_avg_salary",
    avg("salary").over(avg_window)
).show()


+---+------+----------+------+----------+---------------+
| id|  name|department|salary| join_date|dept_avg_salary|
+---+------+----------+------+----------+---------------+
|  4| Priya|   Finance| 65000|2023-02-10|        65000.0|
|  2|  Neha|        HR| 50000|2023-01-15|        51000.0|
|  6| Kiran|        HR| 52000|2023-03-18|        51000.0|
|  1|  Amit|        IT| 60000|2023-01-10|        70000.0|
|  3|   Raj|        IT| 70000|2023-02-01|        70000.0|
|  5|Suresh|        IT| 80000|2023-03-05|        70000.0|
+---+------+----------+------+----------+---------------+



## Spark SQL Window Functions

In [ ]:

df.createOrReplaceTempView("employees")

spark.sql("""
SELECT *,
       ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rn,
       AVG(salary) OVER (PARTITION BY department) AS dept_avg_salary
FROM employees
""").show()


+---+------+----------+------+----------+---+---------------+
| id|  name|department|salary| join_date| rn|dept_avg_salary|
+---+------+----------+------+----------+---+---------------+
|  4| Priya|   Finance| 65000|2023-02-10|  1|        65000.0|
|  6| Kiran|        HR| 52000|2023-03-18|  1|        51000.0|
|  2|  Neha|        HR| 50000|2023-01-15|  2|        51000.0|
|  5|Suresh|        IT| 80000|2023-03-05|  1|        70000.0|
|  3|   Raj|        IT| 70000|2023-02-01|  2|        70000.0|
|  1|  Amit|        IT| 60000|2023-01-10|  3|        70000.0|
+---+------+----------+------+----------+---+---------------+



## End of Advanced Spark Notebook